# ANCHOR External Solver Template

This notebook reads an ANCHOR solver packet exported by the browser game, computes a waypoint plan, and writes `anchor.plan.json` for import back into the game.

**Colab proposes. Game validates. Game simulates. Game scores.**

This is a planning mirror, not the authoritative simulator. It does not control the browser game and it does not replace ANCHOR route validation or scoring.

## 1. Setup

If you are running this notebook inside a cloned copy of the repository, the helper import should work as-is. In Colab, upload this notebook from the repository or clone/copy the `tools/python/anchor_headless` folder next to it.

In [ ]:
from pathlib import Path
import sys

# Local repo path fallback. Adjust this if you copy the notebook elsewhere.
REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / 'tools' / 'python' / 'anchor_headless').exists():
        REPO_ROOT = parent
        break

HELPER_ROOT = REPO_ROOT / 'tools' / 'python'
if str(HELPER_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPER_ROOT))

from anchor_headless import (
    build_headless_world,
    build_plan_json,
    greedy_forecast_plan,
    load_solver_packet,
    sanity_check_plan,
    summarize_packet,
    write_plan_json,
)

print('Helper root:', HELPER_ROOT)

## 2. Upload or Locate Solver Packet

In ANCHOR, open Planning and choose **Export Solver Packet**. Upload the exported `anchor.solver-packet.json` here, or set `SOLVER_PACKET_PATH` to a local file path.

In [ ]:
SOLVER_PACKET_PATH = 'anchor.solver-packet.json'

try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if uploaded:
        SOLVER_PACKET_PATH = next(iter(uploaded.keys()))
except Exception:
    print('Colab upload is unavailable; using local path:', SOLVER_PACKET_PATH)

packet = load_solver_packet(SOLVER_PACKET_PATH)
print('Loaded packet:', SOLVER_PACKET_PATH)

## 3. Inspect Packet Summary

The default workflow is forecast-only and non-oracle. Hidden truth is not used by this template solver.

In [ ]:
summary = summarize_packet(packet)
for key, value in summary.items():
    print(f'{key}: {value}')

if summary.get('oracleMode'):
    print('\nWARNING: Oracle mode is for benchmarking/research only. Do not compare oracle-assisted plans as fair leaderboard entries.')

## 4. Build a Headless Planning World

This lightweight world includes grid size, terrain, hazards/depth when visible, forecast currents/ROI when available, mission timing, agent specs, deployment choices, and scoring metadata. It intentionally does not load hidden truth in the default path.

In [ ]:
world = build_headless_world(packet)
print(f"Grid: {world['width']} x {world['height']}")
print(f"Duration: {world['duration']} | planning window: {world['planningWindow']} | windows: {world['windowCount']}")
print(f"Agents: {len(world['agents'])}")
print(f"ROI rows visible: {len(world['roi'])}")
print(f"Current rows visible: {len(world['current'])}")

## 5. Simple Baseline Solver

This starter solver chooses a valid start/deployment position, ranks visible forecast ROI by expected value per distance, rejects out-of-bounds/land/hazard cells, estimates travel time and fuel with a simple distance model, and writes a timed open-loop waypoint list.

This is a starter solver. The browser game remains the official validator.

In [ ]:
agent_plans = greedy_forecast_plan(world, max_waypoints=4)
plan = build_plan_json(packet, agent_plans, planner_label='colab-template-greedy-v1')

waypoint_count = sum(len(agent_plan.get('waypoints') or []) for agent_plan in plan['agentPlans'])
print('Generated waypoints:', waypoint_count)
print('Planner fairness:', plan['planner'])

## 6. Basic Plan Sanity Checks

These checks catch obvious JSON mistakes before export. They do not replace ANCHOR route validation, simulation, or scoring.

In [ ]:
errors = sanity_check_plan(plan, world)
if errors:
    print('Sanity check errors:')
    for error in errors:
        print('-', error)
    raise ValueError('Plan failed notebook sanity checks.')
print('Notebook sanity checks passed. ANCHOR must still validate and score this plan.')

## 7. Export `anchor.plan.json`

The exported plan uses the project plan schema conventions: `type: anchor.plan`, `executionMode: timedOpenLoop`, top-level `planner` fairness metadata, and `agentPlans[].waypoints`.

In [ ]:
output_path = write_plan_json(plan, 'anchor.plan.json')
print('Wrote', output_path)

try:
    from google.colab import files  # type: ignore
    files.download(str(output_path))
except Exception:
    print('Download helper unavailable; file is written locally.')

## 8. Import Back Into ANCHOR

In ANCHOR:

1. Click **Import Plan**.
2. Select `anchor.plan.json`.
3. Review route validation.
4. Press Play only after validation passes.
5. Export result after simulation if desired.

Colab proposes. Game validates. Game simulates. Game scores.

## Optional: Run the JavaScript Headless Solver With Node.js

The repository also includes a Node.js solver path that imports portable ANCHOR core JavaScript modules. This avoids Python translation drift for route checks and plan contracts.

Use this option when the notebook is running from a cloned ANCHOR repository or when `tools/js` and `src/core` are available in the notebook filesystem. It still does not control the browser game.

Node proposes. Game validates. Game simulates. Game scores.

In [ ]:
import subprocess

# Requires the ANCHOR repository files, including tools/js and src/core.
# In Colab, clone the repo or upload/copy those folders before running this cell.
subprocess.run([
    'node',
    str(REPO_ROOT / 'tools' / 'js' / 'headless_solver.mjs'),
    SOLVER_PACKET_PATH,
    'anchor.plan.json',
    '--planner',
    'greedy'
], check=True)

print('Node headless solver wrote anchor.plan.json')

## 9. Surface Update Stub

Future adaptive workflow:

`anchor.surface-observation.json` -> `anchor.plan-segment.json`

1. Game reaches a surface/update window.
2. Game exports a surface observation.
3. Notebook computes the next segment from the actual surfaced position.
4. Notebook exports `anchor.plan-segment.json`.
5. Game imports, validates, replaces future waypoints, and continues.

This notebook does not implement a complex adaptive solver yet; the section is here to show the intended JSON contract.